In [ ]:
import requests
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import time
from datetime import datetime 
import os
import json
import pathlib as ptb
from sqlalchemy import create_engine

In [2]:
load_dotenv()
api_key = os.getenv('API_TOKEN')


headers = {'X-Auth-Token': api_key}
base_url = 'https://api.football-data.org/v4'

### Sacar los parametros del API

In [4]:
url_m = f"{base_url}/competitions/PL/matches"
params_m = {"season": 2023}

response = requests.get(url_m, headers=headers, params=params_m)

print(response.status_code)

disponibles = response.headers.get("X-Requests-Available-Minute")
reset_seg = response.headers.get("X-RequestCounter-Reset")

print(disponibles, reset_seg)

print(dict(response.headers))

200
9 60
{'Server': 'nginx', 'Date': 'Thu, 25 Jun 2026 23:23:30 GMT', 'Content-Type': 'application/json;charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'X-Requests-Available-Minute': '9', 'X-RequestCounter-Reset': '60', 'X-API-Version': 'v4', 'X-Authenticated-Client': 'brger', 'Access-Control-Allow-Origin': 'http://localhost', 'Access-Control-Allow-Methods': 'GET', 'Content-Language': 'en-US', 'Content-Encoding': 'gzip'}


### Conocer las competiciones del API

In [ ]:
url_c = f'{base_url}/competitions'
response = requests.get(url_c,headers=headers)
data = response.json()

print(json.dumps(data,indent=2)) #Ordenar visualmente los datos del API - json

{
  "count": 13,
  "filters": {
    "client": "brger"
  },
  "competitions": [
    {
      "id": 2013,
      "area": {
        "id": 2032,
        "name": "Brazil",
        "code": "BRA",
        "flag": "https://crests.football-data.org/764.svg"
      },
      "name": "Campeonato Brasileiro S\u00e9rie A",
      "code": "BSA",
      "type": "LEAGUE",
      "emblem": "https://crests.football-data.org/bsa.png",
      "plan": "TIER_ONE",
      "currentSeason": {
        "id": 2474,
        "startDate": "2026-01-28",
        "endDate": "2026-12-02",
        "currentMatchday": 19,
        "winner": null
      },
      "numberOfAvailableSeasons": 10,
      "lastUpdated": "2024-09-13T16:55:53Z"
    },
    {
      "id": 2016,
      "area": {
        "id": 2072,
        "name": "England",
        "code": "ENG",
        "flag": "https://crests.football-data.org/770.svg"
      },
      "name": "Championship",
      "code": "ELC",
      "type": "LEAGUE",
      "emblem": "https://crests.football-da

In [6]:
for comp in data['competitions']:
    print(comp['code'],'-',comp['name'],'-',comp['area']['name'])


BSA - Campeonato Brasileiro Série A - Brazil
ELC - Championship - England
PL - Premier League - England
CL - UEFA Champions League - Europe
EC - European Championship - Europe
FL1 - Ligue 1 - France
BL1 - Bundesliga - Germany
SA - Serie A - Italy
DED - Eredivisie - Netherlands
PPL - Primeira Liga - Portugal
CLI - Copa Libertadores - South America
PD - Primera Division - Spain
WC - FIFA World Cup - World


### saber las temporadas | Ligas disponibles con el API

In [ ]:
def probar_temporada(liga, year):
    url = f"{base_url}/competitions/{liga}/matches"
    response = requests.get(url, headers=headers, params={"season": year})
    
    #Obtenemos las declaraciones de intentos de llamadas restantes y tiempo de actualizacion
    disponibles = int(response.headers.get("X-Requests-Available-Minute"))
    reset_seg = int(response.headers.get("X-RequestCounter-Reset"))
    
    # si las llamadas estan al limite para bloquear descansa el tiempo indicado por reset_seg
    if disponibles <= 1:
        print(f"Quedan {disponibles} llamadas, esperando {reset_seg}s...")
        time.sleep(reset_seg)
    
    return response.status_code

In [15]:
def temporadas_ligas_disponibles(*ligas, year_final=2026, year_inicial=2015):
    temporadas_validas = {}
    
    for liga in ligas:
        # Inicializamos una lista vacía para guardar los años válidos de ESTA liga
        temporadas_validas[liga] = []
        
        for year in range(year_final, year_inicial - 1, -1):
            status = probar_temporada(liga, year)
            
            if status == 200:
                # Guardamos el año exitoso dentro de la lista de la liga
                temporadas_validas[liga].append(year)
                print(f'{liga} - {year} : OK')
            else:
                print(f'{liga} - {year} : NOK')
                
    # El return debe estar AL FINAL de ambos bucles (alineado con el primer for)
    return temporadas_validas

In [3]:
ligas_eu = ["PL", "PD", "BL1", "SA", "FL1"]

In [ ]:
temporadas_ligas_disponibles(*ligas_eu)

Quedan 0 llamadas, esperando 1s...
PL - 2026 : OK
PL - 2025 : OK
PL - 2024 : OK
PL - 2023 : OK
PL - 2022 : NOK
PL - 2021 : NOK
PL - 2020 : NOK
PL - 2019 : NOK
PL - 2018 : NOK
PL - 2017 : NOK
PL - 2016 : NOK
PL - 2015 : NOK
PD - 2026 : NOK
PD - 2025 : OK
PD - 2024 : OK
PD - 2023 : OK
PD - 2022 : NOK
PD - 2021 : NOK
PD - 2020 : NOK
PD - 2019 : NOK
PD - 2018 : NOK
PD - 2017 : NOK
PD - 2016 : NOK
PD - 2015 : NOK
BL1 - 2026 : NOK
BL1 - 2025 : OK
BL1 - 2024 : OK
BL1 - 2023 : OK
Quedan 1 llamadas, esperando 27s...
BL1 - 2022 : NOK
BL1 - 2021 : NOK
BL1 - 2020 : NOK
BL1 - 2019 : NOK
BL1 - 2018 : NOK
BL1 - 2017 : NOK
BL1 - 2016 : NOK
BL1 - 2015 : NOK
SA - 2026 : OK
SA - 2025 : OK
SA - 2024 : OK
SA - 2023 : OK
SA - 2022 : NOK
SA - 2021 : NOK
SA - 2020 : NOK
SA - 2019 : NOK
SA - 2018 : NOK
SA - 2017 : NOK
SA - 2016 : NOK
SA - 2015 : NOK
FL1 - 2026 : OK
FL1 - 2025 : OK
FL1 - 2024 : OK
FL1 - 2023 : OK
FL1 - 2022 : NOK
FL1 - 2021 : NOK
FL1 - 2020 : NOK
FL1 - 2019 : NOK
FL1 - 2018 : NOK
FL1 - 2017 : N

{'PL': [2026, 2025, 2024, 2023],
 'PD': [2025, 2024, 2023],
 'BL1': [2025, 2024, 2023],
 'SA': [2026, 2025, 2024, 2023],
 'FL1': [2026, 2025, 2024, 2023]}

In [4]:
# como la funcion no lo corri dentro de una variable ademas de tener los print(OK/NOK)
# no se guardaron por ello este bloque de resultados

resultado_temporadas = {
    'PL': [2026, 2025, 2024, 2023],
    'PD': [2025, 2024, 2023],
    'BL1': [2025, 2024, 2023],
    'SA': [2026, 2025, 2024, 2023],
    'FL1': [2026, 2025, 2024, 2023]
}

carpeta = ptb.Path("../data/raw")
carpeta.mkdir(parents=True, exist_ok=True)  # crea la carpeta si no existe

with open(carpeta / "temporadas_disponibles.json", "w", encoding="utf-8") as f:
    json.dump(resultado_temporadas, f, indent=2)

### Descargar los partidos de Liga | Temporada

In [12]:
def descargar_temp(liga,year):
    base_url = f'https://api.football-data.org/v4/competitions/{liga}/matches'
    param = {'season':year}

    response = requests.get(base_url,headers = headers,params = param)
    intento = int(response.headers.get('X-Requests-Available-Minute'))
    tiempo = int(response.headers.get('X-RequestCounter-Reset'))

    carpeta = ptb.Path('../data/raw')
    carpeta.mkdir(parents = True,exist_ok = True)
    if response.status_code == 200:
        with open(carpeta / f'{liga}_{year}.json','w',encoding='utf-8') as f:
            json.dump(response.json(),f,indent= 2)
    else:
        pass
    
    if intento < 2:
        time.sleep(tiempo)

In [13]:
anios_historico = [2023, 2024, 2025]

for liga, años in resultado_temporadas.items():
    for anio in años:
        if anio in anios_historico:
            print(f'Descargando {liga}: {anio}...')
            descargar_temp(liga, anio)

Descargando PL: 2025...
Descargando PL: 2024...
Descargando PL: 2023...
Descargando PD: 2025...
Descargando PD: 2024...
Descargando PD: 2023...
Descargando BL1: 2025...
Descargando BL1: 2024...
Descargando BL1: 2023...
Descargando SA: 2025...
Descargando SA: 2024...
Descargando SA: 2023...
Descargando FL1: 2025...
Descargando FL1: 2024...
Descargando FL1: 2023...


In [14]:
carpeta = ptb.Path('../data/raw')
archivos = sorted(carpeta.glob('*.json'))
print(f"Total archivos: {len(archivos)}")
for a in archivos:
    print(a.name, "-", round(a.stat().st_size / 1024, 1), "KB")

Total archivos: 16
BL1_2023.json - 646.5 KB
BL1_2024.json - 648.5 KB
BL1_2025.json - 508.5 KB
FL1_2023.json - 507.9 KB
FL1_2024.json - 508.7 KB
FL1_2025.json - 506.8 KB
PD_2023.json - 637.3 KB
PD_2024.json - 637.9 KB
PD_2025.json - 632.5 KB
PL_2023.json - 633.1 KB
pl_2024.json - 633.8 KB
PL_2025.json - 632.9 KB
SA_2023.json - 626.3 KB
SA_2024.json - 625.5 KB
SA_2025.json - 626.1 KB
temporadas_disponibles.json - 0.3 KB


In [15]:
import json
with open('../data/raw/BL1_2023.json', encoding='utf-8') as f:
    data = json.load(f)
print("Claves:", data.keys())
print("Cantidad de partidos:", len(data['matches']))
print(data['matches'][0]['homeTeam']['name'], "vs", data['matches'][0]['awayTeam']['name'])

Claves: dict_keys(['filters', 'resultSet', 'competition', 'matches'])
Cantidad de partidos: 306
SV Werder Bremen vs FC Bayern München


### Cargar a BD con info de los partidos

In [3]:
carpeta = ptb.Path('../data/raw')
archivos = sorted(carpeta.glob('*.json'))
listas_dfs = []

for b in archivos:

    nombre = b.stem

    if b.name != "temporadas_disponibles.json":

        liga = nombre.split("_")[0]
        anio = nombre.split("_")[1]

        with open(b,encoding='utf-8') as f:
            d = json.load(f)
        
        d1 = pd.json_normalize(d['matches'])
        d1['Liga_archivo'] = liga
        d1['temporada_archivo'] = anio

        listas_dfs.append(d1)


partidos = pd.concat(listas_dfs,ignore_index=True)


### EDA Partidos

In [4]:
print("Total de filas:", len(partidos))
print("Columnas:", partidos.columns.tolist())
print("\nPartidos por liga y temporada:")
print(partidos.groupby(['Liga_archivo', 'temporada_archivo']).size())

Total de filas: 5256
Columnas: ['id', 'utcDate', 'status', 'matchday', 'stage', 'group', 'lastUpdated', 'referees', 'area.id', 'area.name', 'area.code', 'area.flag', 'competition.id', 'competition.name', 'competition.code', 'competition.type', 'competition.emblem', 'season.id', 'season.startDate', 'season.endDate', 'season.currentMatchday', 'season.winner.id', 'season.winner.name', 'season.winner.shortName', 'season.winner.tla', 'season.winner.crest', 'season.winner.address', 'season.winner.website', 'season.winner.founded', 'season.winner.clubColors', 'season.winner.venue', 'season.winner.lastUpdated', 'homeTeam.id', 'homeTeam.name', 'homeTeam.shortName', 'homeTeam.tla', 'homeTeam.crest', 'awayTeam.id', 'awayTeam.name', 'awayTeam.shortName', 'awayTeam.tla', 'awayTeam.crest', 'score.winner', 'score.duration', 'score.fullTime.home', 'score.fullTime.away', 'score.halfTime.home', 'score.halfTime.away', 'odds.msg', 'Liga_archivo', 'temporada_archivo', 'season.winner']

Partidos por liga y 

In [5]:
partidos.columns

Index(['id', 'utcDate', 'status', 'matchday', 'stage', 'group', 'lastUpdated',
       'referees', 'area.id', 'area.name', 'area.code', 'area.flag',
       'competition.id', 'competition.name', 'competition.code',
       'competition.type', 'competition.emblem', 'season.id',
       'season.startDate', 'season.endDate', 'season.currentMatchday',
       'season.winner.id', 'season.winner.name', 'season.winner.shortName',
       'season.winner.tla', 'season.winner.crest', 'season.winner.address',
       'season.winner.website', 'season.winner.founded',
       'season.winner.clubColors', 'season.winner.venue',
       'season.winner.lastUpdated', 'homeTeam.id', 'homeTeam.name',
       'homeTeam.shortName', 'homeTeam.tla', 'homeTeam.crest', 'awayTeam.id',
       'awayTeam.name', 'awayTeam.shortName', 'awayTeam.tla', 'awayTeam.crest',
       'score.winner', 'score.duration', 'score.fullTime.home',
       'score.fullTime.away', 'score.halfTime.home', 'score.halfTime.away',
       'odds.msg', 'L

In [6]:
partidos['status'].unique()

array(['FINISHED', 'AWARDED'], dtype=object)

In [7]:
partidos[partidos['status']== 'AWARDED']

,id,utcDate,status,matchday,stage,group,lastUpdated,referees,area.id,area.name,...,score.winner,score.duration,score.fullTime.home,score.fullTime.away,score.halfTime.home,score.halfTime.away,odds.msg,Liga_archivo,temporada_archivo,season.winner
426,502494,2024-12-14T14:30:00Z,AWARDED,14,REGULAR_SEASON,None,2025-06-27T16:59:01Z,"[{'id': 15825, 'name': 'Martin Petersen', 'typ...",2088,Germany,...,AWAY_TEAM,REGULAR,0,2,NaN,NaN,Activate Odds-Package in User-Panel to retriev...,BL1,2024,NaN
1454,498178,2025-03-16T16:15:00Z,AWARDED,26,REGULAR_SEASON,None,2025-06-01T20:20:47Z,"[{'id': 43918, 'name': 'François Letexier', 't...",2081,France,...,AWAY_TEAM,REGULAR,0,2,0.0,1.0,Activate Odds-Package in User-Panel to retriev...,FL1,2024,None
1828,542704,2026-05-17T19:00:00Z,AWARDED,34,REGULAR_SEASON,None,2026-05-30T20:20:41Z,"[{'id': 25786, 'name': 'Stéphanie Frappart', '...",2081,France,...,DRAW,REGULAR,0,0,NaN,NaN,Activate Odds-Package in User-Panel to retriev...,FL1,2025,None


se tienen 2 status de partidos:*FINISHED* y *AWARDED*

Esto es porque hay partidos llevados a procesos administrativos el cual con lleva a decidir el resultado del partido en los despachos

In [8]:
(partidos.isnull().sum()/len(partidos))*100

id                             0.000000
utcDate                        0.000000
status                         0.000000
matchday                       0.000000
stage                          0.000000
group                        100.000000
lastUpdated                    0.000000
referees                       0.000000
area.id                        0.000000
area.name                      0.000000
area.code                      0.000000
area.flag                      0.000000
competition.id                 0.000000
competition.name               0.000000
competition.code               0.000000
competition.type               0.000000
competition.emblem             0.000000
season.id                      0.000000
season.startDate               0.000000
season.endDate                 0.000000
season.currentMatchday         0.000000
season.winner.id              88.356164
season.winner.name            88.356164
season.winner.shortName       88.356164
season.winner.tla             88.356164


In [9]:
partidos[partidos['score.halfTime.away'].isnull() == True]

,id,utcDate,status,matchday,stage,group,lastUpdated,referees,area.id,area.name,...,score.winner,score.duration,score.fullTime.home,score.fullTime.away,score.halfTime.home,score.halfTime.away,odds.msg,Liga_archivo,temporada_archivo,season.winner
426,502494,2024-12-14T14:30:00Z,AWARDED,14,REGULAR_SEASON,None,2025-06-27T16:59:01Z,"[{'id': 15825, 'name': 'Martin Petersen', 'typ...",2088,Germany,...,AWAY_TEAM,REGULAR,0,2,NaN,NaN,Activate Odds-Package in User-Panel to retriev...,BL1,2024,NaN
1828,542704,2026-05-17T19:00:00Z,AWARDED,34,REGULAR_SEASON,None,2026-05-30T20:20:41Z,"[{'id': 25786, 'name': 'Stéphanie Frappart', '...",2081,France,...,DRAW,REGULAR,0,0,NaN,NaN,Activate Odds-Package in User-Panel to retriev...,FL1,2025,None


Existen resultados de goles a mitad de tiempo con nulos pero es debido a lo categorizado por *AWARDED*

In [10]:
partidos[['lastUpdated','utcDate']].head(3)

,lastUpdated,utcDate
0,2024-09-29T10:21:38Z,2023-08-18T18:30:00Z
1,2024-09-29T10:21:38Z,2023-08-19T13:30:00Z
2,2024-09-29T10:21:38Z,2023-08-19T13:30:00Z


In [11]:
partidos['lastUpdated'] = pd.to_datetime(partidos['lastUpdated'])
partidos['utcDate'] = pd.to_datetime(partidos['utcDate'])

In [12]:
print(partidos[['lastUpdated', 'utcDate']].dtypes)

lastUpdated    datetime64[ns, UTC]
utcDate        datetime64[ns, UTC]
dtype: object


In [13]:
partidos[['lastUpdated','utcDate']].head(3)

,lastUpdated,utcDate
0,2024-09-29 10:21:38+00:00,2023-08-18 18:30:00+00:00
1,2024-09-29 10:21:38+00:00,2023-08-19 13:30:00+00:00
2,2024-09-29 10:21:38+00:00,2023-08-19 13:30:00+00:00


se formateo las fechas horas para poder hacer calculo con ellas

In [14]:
dif = partidos['lastUpdated']- partidos['utcDate']
dif_dias = dif.dt.days  
np.percentile(dif_dias,90)

np.float64(265.0)

se puede ver que existio una actualizacion masiva de todos los datos que ya estaban cargados en el API por ello lleva a tener un registro muy alto de dias transcurridos entre el partido emitido y la fecha de actualizacion

### Revisando la columna "referees"

In [20]:
partidos['n_arbitros'] = partidos['referees'].apply(len)
print(partidos['n_arbitros'].value_counts(normalize=True)*100)

n_arbitros
1    99.162861
0     0.722983
2     0.114155
Name: proportion, dtype: float64


In [21]:
partidos

,id,utcDate,status,matchday,stage,group,lastUpdated,referees,area.id,area.name,...,score.duration,score.fullTime.home,score.fullTime.away,score.halfTime.home,score.halfTime.away,odds.msg,Liga_archivo,temporada_archivo,season.winner,n_arbitros
0,441789,2023-08-18 18:30:00+00:00,FINISHED,1,REGULAR_SEASON,None,2024-09-29 10:21:38+00:00,"[{'id': 43878, 'name': 'Felix Zwayer', 'type':...",2088,Germany,...,REGULAR,0,4,0.0,1.0,Activate Odds-Package in User-Panel to retriev...,BL1,2023,NaN,1
1,441792,2023-08-19 13:30:00+00:00,FINISHED,1,REGULAR_SEASON,None,2024-09-29 10:21:38+00:00,"[{'id': 43875, 'name': 'Felix Brych', 'type': ...",2088,Germany,...,REGULAR,3,2,2.0,1.0,Activate Odds-Package in User-Panel to retriev...,BL1,2023,NaN,1
2,441794,2023-08-19 13:30:00+00:00,FINISHED,1,REGULAR_SEASON,None,2024-09-29 10:21:38+00:00,"[{'id': 57518, 'name': 'Florian Badstübner', '...",2088,Germany,...,REGULAR,2,0,2.0,0.0,Activate Odds-Package in User-Panel to retriev...,BL1,2023,NaN,1
3,441795,2023-08-19 13:30:00+00:00,FINISHED,1,REGULAR_SEASON,None,2024-09-29 10:21:38+00:00,"[{'id': 15824, 'name': 'Sven Jablonski', 'type...",2088,Germany,...,REGULAR,1,2,0.0,2.0,Activate Odds-Package in User-Panel to retriev...,BL1,2023,NaN,1
4,441796,2023-08-19 13:30:00+00:00,FINISHED,1,REGULAR_SEASON,None,2024-09-29 10:21:38+00:00,"[{'id': 57517, 'name': 'Daniel Schlager', 'typ...",2088,Germany,...,REGULAR,4,4,3.0,3.0,Activate Odds-Package in User-Panel to retriev...,BL1,2023,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5251,537186,2026-05-24 18:45:00+00:00,FINISHED,38,REGULAR_SEASON,None,2026-06-07 20:20:25+00:00,"[{'id': 11079, 'name': 'Marco Guida', 'type': ...",2114,Italy,...,REGULAR,1,2,1.0,1.0,Activate Odds-Package in User-Panel to retriev...,SA,2025,None,1
5252,537189,2026-05-24 18:45:00+00:00,FINISHED,38,REGULAR_SEASON,None,2026-06-07 20:20:25+00:00,"[{'id': 57826, 'name': 'Simone Sozza', 'type':...",2114,Italy,...,REGULAR,0,2,0.0,0.0,Activate Odds-Package in User-Panel to retriev...,SA,2025,None,1
5253,537193,2026-05-24 18:45:00+00:00,FINISHED,38,REGULAR_SEASON,None,2026-06-07 20:20:25+00:00,"[{'id': 10977, 'name': 'Fabio Maresca', 'type'...",2114,Italy,...,REGULAR,1,4,0.0,1.0,Activate Odds-Package in User-Panel to retriev...,SA,2025,None,1
5254,537194,2026-05-24 18:45:00+00:00,FINISHED,38,REGULAR_SEASON,None,2026-06-07 20:20:25+00:00,"[{'id': 11065, 'name': 'Daniele Doveri', 'type...",2114,Italy,...,REGULAR,1,0,1.0,0.0,Activate Odds-Package in User-Panel to retriev...,SA,2025,None,1


### Prueba de conexion con SQL SERVER

In [23]:
db_server = os.getenv('DB_SERVER')   # BRANDON\SQLEXPRESS

connection_string = (
    f"mssql+pyodbc://@{db_server}/football_dwh"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

In [24]:
from sqlalchemy import text
#Probando la conexion a la base de datos
with engine.connect() as conn:
    resultado = conn.execute(text("SELECT @@VERSION"))
    print(resultado.fetchone()[0]) 

Microsoft SQL Server 2025 (RTM-GDR) (KB5091223) - 17.0.1115.1 (X64) 
	Apr 19 2026 01:00:58 
	Copyright (C) 2025 Microsoft Corporation
	Express Edition (64-bit) on Windows 10 Home Single Language 10.0 <X64> (Build 26200: ) (Hypervisor)



C:\Users\brger\AppData\Local\Temp\ipykernel_10596\1070122320.py:3: SAWarning: Unrecognized server version info '17.0.1115.1'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


In [26]:
partidos['referees'] = partidos['referees'].apply(json.dumps)  # Convertir listas a cadenas JSON

In [27]:
partidos['fecha_carga'] = datetime.now()

In [28]:
prueba = partidos[(partidos['Liga_archivo']=='PL') & (partidos['temporada_archivo']=='2023')]

prueba.to_sql('partidos', con=engine, schema='bronze', if_exists='replace', index=False)

ProgrammingError: (pyodbc.ProgrammingError) ('42000', "[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]A table can only have one timestamp column. Because table 'partidos' already has one, the column 'lastUpdated' cannot be added. (2738) (SQLExecDirectW)")
[SQL: 
CREATE TABLE bronze.partidos (
	id BIGINT NULL, 
	[utcDate] TIMESTAMP NULL, 
	status VARCHAR(max) NULL, 
	matchday BIGINT NULL, 
	stage VARCHAR(max) NULL, 
	[group] VARCHAR(max) NULL, 
	[lastUpdated] TIMESTAMP NULL, 
	referees VARCHAR(max) NULL, 
	[area.id] BIGINT NULL, 
	[area.name] VARCHAR(max) NULL, 
	[area.code] VARCHAR(max) NULL, 
	[area.flag] VARCHAR(max) NULL, 
	[competition.id] BIGINT NULL, 
	[competition.name] VARCHAR(max) NULL, 
	[competition.code] VARCHAR(max) NULL, 
	[competition.type] VARCHAR(max) NULL, 
	[competition.emblem] VARCHAR(max) NULL, 
	[season.id] BIGINT NULL, 
	[season.startDate] VARCHAR(max) NULL, 
	[season.endDate] VARCHAR(max) NULL, 
	[season.currentMatchday] BIGINT NULL, 
	[season.win

In [29]:
from sqlalchemy.types import DateTime, NVARCHAR

prueba.to_sql('partidos', con=engine, schema='bronze', 
              if_exists='replace', index=False,
              dtype={
                  'utcDate': NVARCHAR(50),
                  'lastUpdated': NVARCHAR(50),
                  'season.startDate': NVARCHAR(50),
                  'season.endDate': NVARCHAR(50),
                  'fecha_carga': DateTime()
              })

38

In [30]:
check = pd.read_sql("SELECT COUNT(*) as total FROM bronze.partidos", engine)
print(check)

   total
0    380


In [31]:
partidos.to_sql('partidos', con=engine, schema='bronze',
                if_exists='replace', index=False,
                dtype={
                    'utcDate': NVARCHAR(50),
                    'lastUpdated': NVARCHAR(50),
                    'season.startDate': NVARCHAR(50),
                    'season.endDate': NVARCHAR(50),
                    'fecha_carga': DateTime()
                })

12

In [32]:
check = pd.read_sql("SELECT COUNT(*) as total FROM bronze.partidos", engine)
print(check)   # debería dar 5256

   total
0   5256
